# SQA Benchmark — FPGA side

Run this notebook **on the PYNQ-Z2 board**.

It sweeps over a range of problem sizes, solves each with the FPGA kernel,
and saves the results to `fpga_results.csv`.

**After running:** copy `fpga_results.csv` to your laptop and open
`SQA-Benchmark-Compare.ipynb` alongside `laptop_results.csv`.

**Files required:**
```
/home/xilinx/jupyter_notebooks/Quantum Annealing Simulation/SQA_Opt5.bit
/home/xilinx/jupyter_notebooks/Quantum Annealing Simulation/SQA_Opt5.hwh
/home/xilinx/jupyter_notebooks/Quantum Annealing Simulation/sqa_solver.py
```

In [ ]:
import sys
sys.path.append('/home/xilinx/jupyter_notebooks/Quantum Annealing Simulation')

import numpy as np
import pandas as pd
from sqa_solver import SQASolver

In [ ]:
BITFILE = '/home/xilinx/jupyter_notebooks/Quantum Annealing Simulation/SQA_Opt5.bit'
solver = SQASolver(BITFILE, num_trotters=8)
print('Overlay loaded.')

In [ ]:
# -- Benchmark configuration -------------------------------------------------
# Must match exactly what is used in SQA-Benchmark-Laptop.ipynb.
SIZES       = [8, 16, 32, 64, 128, 256, 512, 1024]
ITERS       = 200
RESTARTS    = 3     # multiple restarts improve quality without long oscillating runs
BETA_START  = 1 / 4096
BETA_END    = 8.0
GAMMA_START = 8.0
SEED        = 0

OUTPUT_CSV  = 'fpga_results.csv'


In [ ]:
def make_qubo(n, seed):
    """Deterministic random dense symmetric QUBO -- must be identical in both notebooks."""
    rng = np.random.default_rng(seed)
    Q = rng.uniform(-1, 1, size=(n, n))
    Q = (Q + Q.T) / 2
    np.fill_diagonal(Q, rng.uniform(-2, 0, size=n))
    return Q.astype(np.float32)


schedule = dict(iters=ITERS, restarts=RESTARTS, beta_start=BETA_START,
                beta_end=BETA_END, gamma_start=GAMMA_START)

rows = []

for n in SIZES:
    Q = make_qubo(n, SEED)
    print(f'n={n:4d} ... ', end='', flush=True)

    r = solver.solve(Q, seed=SEED, **schedule)

    rows.append(dict(
        n             = n,
        time_s        = r.timing_s,
        best_energy   = r.best_energy,
        trotter_mean  = float(np.mean(r.all_energies)),
        trotter_std   = float(np.std(r.all_energies)),
    ))
    print(f'done  {r.timing_s:.4f} s   energy {r.best_energy:.4f}')

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved {OUTPUT_CSV}')
display(df)


---
## 2a — Iteration sweep (restarts=1, shows oscillation problem)

With a single restart and a slow schedule, the Jacobi-update kernel oscillates  
at high iteration counts and quality **degrades**. This is a known kernel limitation.

In [ ]:
ITER_SWEEP_N    = 64
ITER_SWEEP_VALS = [50, 100, 200, 500, 1000, 2000, 5000]
ITER_SWEEP_CSV  = 'fpga_iter_sweep.csv'
Q_sw = make_qubo(ITER_SWEEP_N, SEED)

iter_rows = []
print(f'Iteration sweep  n={ITER_SWEEP_N}  restarts=1')
print(f'{"iters":>6}  {"time (s)":>10}  {"ms/iter":>8}  {"best energy":>14}')
print('-' * 46)

for n_iter in ITER_SWEEP_VALS:
    r = solver.solve(Q_sw, seed=SEED, iters=n_iter, restarts=1,
                     beta_start=BETA_START, beta_end=BETA_END,
                     gamma_start=GAMMA_START)
    ms_per = r.timing_s / n_iter * 1000
    iter_rows.append(dict(iters=n_iter, time_s=r.timing_s, best_energy=r.best_energy))
    print(f'{n_iter:>6}  {r.timing_s:>10.4f}  {ms_per:>8.2f}  {r.best_energy:>14.4f}')

df_iter = pd.DataFrame(iter_rows)
df_iter.to_csv(ITER_SWEEP_CSV, index=False)
print(f'\nSaved {ITER_SWEEP_CSV}')
display(df_iter)


---
## 2b — Restarts sweep (iters=200, the actual fix)

Keep `iters=200` (the sweet spot for the Jacobi kernel) and increase the number  
of independent restarts instead. Each restart re-randomises the trotters and runs  
a full fresh schedule. Because the FPGA is fast, many restarts are cheap.

In [ ]:
RESTART_VALS     = [1, 2, 3, 5, 10]
RESTART_SWEEP_CSV = 'fpga_restart_sweep.csv'

restart_rows = []
print(f'Restarts sweep  n={ITER_SWEEP_N}  iters=200')
print(f'{"restarts":>9}  {"time (s)":>10}  {"best energy":>14}')
print('-' * 40)

for nr in RESTART_VALS:
    r = solver.solve(Q_sw, seed=SEED, iters=200, restarts=nr,
                     beta_start=BETA_START, beta_end=BETA_END,
                     gamma_start=GAMMA_START)
    restart_rows.append(dict(restarts=nr, time_s=r.timing_s, best_energy=r.best_energy))
    print(f'{nr:>9}  {r.timing_s:>10.4f}  {r.best_energy:>14.4f}')

df_restart = pd.DataFrame(restart_rows)
df_restart.to_csv(RESTART_SWEEP_CSV, index=False)
print(f'\nSaved {RESTART_SWEEP_CSV}')
display(df_restart)
